# AnalyticaPuzzle — Error Analysis Pipeline

**Goal:** Sample ~200 failures stratified by model, get per-failure summaries from o3, then run a qualitative taxonomy over all summaries.

**Schema assumed:**
```
Puzzle_Number, Question_Number, Sample, Raw_Output, Reasoning,
Model_Answer, Inference_Min, model_name, audited_answer,
audited_raw, audit_elapsed, GT_Answer, Solved_Percentile,
difficulty, correct, Answer, Original_Answer
```

**Failure definition:** `audited_answer != GT_Answer`  
(We use `audited_answer` as the authoritative answer, not `Model_Answer`, since it has been verified by the judge.)

## 0. Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DF1_PATH = "results_df1.csv"          # <── change to your first results file
DF2_PATH = "results_df2.csv"          # <── change to your second results file

# ── Sampling ──────────────────────────────────────────────────────────────────
TARGET_SAMPLES   = 200                # total failures to sample
RANDOM_SEED      = 42

# ── API ───────────────────────────────────────────────────────────────────────
import os
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")   # set in env or paste here
SUMMARY_MODEL    = "claude-opus-4-5"   # for per-failure summaries  (swap to o3 if via OpenAI)
TAXONOMY_MODEL   = "claude-opus-4-5"   # for final taxonomy

# ── Output ────────────────────────────────────────────────────────────────────
FAILURES_OUT     = "failures_sample.csv"        # 200 failure rows + metadata
SUMMARIES_OUT    = "failure_summaries.csv"       # per-row summaries added
TAXONOMY_OUT     = "error_taxonomy.txt"          # final qualitative report

print("Config loaded.")

## 1. Load & Merge

In [ ]:
import pandas as pd
import numpy as np

df1 = pd.read_csv(DF1_PATH)
df2 = pd.read_csv(DF2_PATH)

df = pd.concat([df1, df2], ignore_index=True)
print(f"Combined rows: {len(df)}")
print(f"Models: {df['model_name'].unique()}")
df.head(2)

## 2. Identify Failures

Failure = `audited_answer != GT_Answer`.  
We normalise both to uppercase stripped strings to avoid casing/whitespace mismatches.

In [ ]:
def normalise(s):
    if pd.isna(s):
        return ""
    return str(s).strip().upper()

df["_audited"] = df["audited_answer"].apply(normalise)
df["_gt"]      = df["GT_Answer"].apply(normalise)

# Flag failures
df["is_failure"] = (df["_audited"] != df["_gt"]) & (df["_audited"] != "") & (df["_gt"] != "")

failures = df[df["is_failure"]].copy()
print(f"Total failures: {len(failures)}")
print(failures.groupby("model_name")["is_failure"].count().rename("n_failures"))

## 3. Stratified Sample (~200 failures, proportional by model)

In [ ]:
def stratified_sample(df_fail, target=TARGET_SAMPLES, seed=RANDOM_SEED):
    """
    Proportional stratified sample by model_name.
    If a stratum has fewer rows than its quota, take all of them.
    """
    counts = df_fail["model_name"].value_counts()
    total  = counts.sum()
    quotas = (counts / total * target).round().astype(int)

    # Fix rounding so total == target
    diff = target - quotas.sum()
    if diff != 0:
        # Add/remove from largest stratum
        quotas.iloc[0] += diff

    parts = []
    for model, q in quotas.items():
        stratum = df_fail[df_fail["model_name"] == model]
        n = min(q, len(stratum))
        parts.append(stratum.sample(n, random_state=seed))

    sample = pd.concat(parts, ignore_index=True)
    return sample


sampled = stratified_sample(failures)
print(f"Sampled: {len(sampled)} rows")
print(sampled.groupby("model_name").size().rename("sampled"))

sampled.to_csv(FAILURES_OUT, index=False)
print(f"Saved to {FAILURES_OUT}")

## 4. Per-Failure Summaries via o3

For each failure, we send:  
- The puzzle question (from `Sample`)  
- The model's reasoning trace (`Reasoning`)  
- The model's answer (`audited_answer`) and the ground truth (`GT_Answer`)  

o3 returns a 2–3 sentence summary: *what specifically went wrong in this trace?*

Results are saved incrementally every 10 rows to `SUMMARIES_OUT`.

In [ ]:
import anthropic
import time

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

SUMMARY_SYSTEM = """\
You are an expert NLP researcher analysing where LLMs fail on analytical reasoning puzzles.
You will be given a puzzle question, the model's reasoning trace, the model's final answer,
and the correct answer.

Write 2–3 sentences describing *what specifically went wrong* in this reasoning trace.
Be concrete: name the exact logical mistake, constraint violation, or reasoning gap.
Do NOT just say "the model got the wrong answer" — explain WHY.
Focus on the failure mode, not on what the correct reasoning should have been.
"""

def build_summary_prompt(row):
    reasoning = str(row.get("Reasoning", "") or row.get("Raw_Output", ""))[:3000]  # cap tokens
    question  = str(row.get("Sample", ""))[:1500]
    return (
        f"**Question:**\n{question}\n\n"
        f"**Model reasoning trace:**\n{reasoning}\n\n"
        f"**Model answer:** {row['audited_answer']}\n"
        f"**Correct answer:** {row['GT_Answer']}\n"
    )

def get_summary(row, retries=3):
    prompt = build_summary_prompt(row)
    for attempt in range(retries):
        try:
            msg = client.messages.create(
                model=SUMMARY_MODEL,
                max_tokens=300,
                system=SUMMARY_SYSTEM,
                messages=[{"role": "user", "content": prompt}]
            )
            return msg.content[0].text.strip()
        except anthropic.RateLimitError:
            time.sleep(20 * (attempt + 1))
        except Exception as e:
            print(f"  Error on attempt {attempt+1}: {e}")
            time.sleep(5)
    return "[SUMMARY_FAILED]"

print("Summary function defined.")

In [ ]:
from pathlib import Path

# ── Resume logic ─────────────────────────────────────────────────────────────
if Path(SUMMARIES_OUT).exists():
    done_df = pd.read_csv(SUMMARIES_OUT)
    done_keys = set(zip(done_df["Puzzle_Number"].astype(str),
                        done_df["Question_Number"].astype(str),
                        done_df["model_name"].astype(str)))
    print(f"Resuming — {len(done_df)} summaries already done.")
else:
    done_df   = pd.DataFrame()
    done_keys = set()

results = []
SAVE_EVERY = 10

for i, (_, row) in enumerate(sampled.iterrows()):
    key = (str(row["Puzzle_Number"]), str(row["Question_Number"]), str(row["model_name"]))
    if key in done_keys:
        continue

    summary = get_summary(row)
    results.append({**row.to_dict(), "error_summary": summary})

    if len(results) % SAVE_EVERY == 0:
        partial = pd.concat([done_df, pd.DataFrame(results)], ignore_index=True)
        partial.to_csv(SUMMARIES_OUT, index=False)
        print(f"  [{i+1}/{len(sampled)}] saved {len(partial)} rows")

# Final save
all_summaries = pd.concat([done_df, pd.DataFrame(results)], ignore_index=True)
all_summaries.to_csv(SUMMARIES_OUT, index=False)
print(f"\nDone. {len(all_summaries)} summaries saved to {SUMMARIES_OUT}")

## 5. Qualitative Taxonomy via o3

We feed all ~200 summaries to o3 in one prompt and ask it to:
1. Identify named error categories with counts
2. Give 2–3 representative examples per category
3. Note any model-specific patterns
4. Produce a brief overall narrative

> **Note on token budget:** 200 × ~200 chars ≈ ~10 k tokens of summaries, well within context.

In [ ]:
all_summaries = pd.read_csv(SUMMARIES_OUT)  # reload in case cell was re-run

# Format summaries for the taxonomy prompt
def format_for_taxonomy(df):
    lines = []
    for i, row in df.iterrows():
        model   = row.get("model_name", "unknown")
        diff    = row.get("difficulty", "?")
        pnum    = row.get("Puzzle_Number", "?")
        qnum    = row.get("Question_Number", "?")
        summary = str(row.get("error_summary", ""))
        lines.append(f"[{i+1}] Model={model} | Difficulty={diff} | P{pnum}{qnum}\n{summary}")
    return "\n\n".join(lines)

taxonomy_input = format_for_taxonomy(all_summaries)
print(f"Input length: {len(taxonomy_input)} chars / ~{len(taxonomy_input)//4} tokens")

In [ ]:
TAXONOMY_SYSTEM = """\
You are an expert NLP researcher performing qualitative error analysis on LLM failures
on analytical reasoning (logic puzzle) tasks.

You will receive a numbered list of error summaries, each describing a specific failure
by an LLM on a puzzle question. Your job is to produce a rigorous qualitative taxonomy.

Output format:
1. **Error Taxonomy Table** — a list of named error categories, count, percentage, and 1-line description.
2. **Per-Category Deep Dive** — for each category: definition, 2–3 representative examples (by index number), and any notable model or difficulty patterns.
3. **Cross-Cutting Patterns** — observations that span multiple categories (e.g., certain models fail more on specific error types, difficulty-level trends, CoT vs non-CoT differences if visible).
4. **Narrative Summary** (1 paragraph) — high-level story of where and why these models struggle.

Be specific, evidence-grounded, and use precise language suitable for an academic paper.
Do not fabricate patterns not visible in the data.
"""

TAXONOMY_USER = f"""\
Below are {len(all_summaries)} error summaries from LLM failures on analytical reasoning puzzles.
Each entry includes the model name, puzzle difficulty, and a description of what went wrong.

{taxonomy_input}

Please produce the full qualitative taxonomy as specified.
"""

print("Sending taxonomy request...")

taxonomy_response = client.messages.create(
    model=TAXONOMY_MODEL,
    max_tokens=4000,
    system=TAXONOMY_SYSTEM,
    messages=[{"role": "user", "content": TAXONOMY_USER}]
)

taxonomy_text = taxonomy_response.content[0].text.strip()

with open(TAXONOMY_OUT, "w") as f:
    f.write(taxonomy_text)

print(f"Taxonomy saved to {TAXONOMY_OUT}")
print("\n" + "="*80 + "\n")
print(taxonomy_text)

## 6. Quick Descriptive Stats (for your manual analysis)

These tables complement the taxonomy and are useful for cross-referencing manually.

In [ ]:
# Failure rate by model
model_stats = df.groupby("model_name").agg(
    total=("is_failure", "count"),
    failures=("is_failure", "sum")
)
model_stats["failure_rate"] = (model_stats["failures"] / model_stats["total"]).round(3)
print("=== Failure Rate by Model ===")
display(model_stats.sort_values("failure_rate", ascending=False))

In [ ]:
# Failure rate by difficulty
diff_stats = df.groupby("difficulty").agg(
    total=("is_failure", "count"),
    failures=("is_failure", "sum")
)
diff_stats["failure_rate"] = (diff_stats["failures"] / diff_stats["total"]).round(3)
print("=== Failure Rate by Difficulty ===")
display(diff_stats)

In [ ]:
# Failure rate by model × difficulty
cross = df.groupby(["model_name", "difficulty"]).agg(
    total=("is_failure", "count"),
    failures=("is_failure", "sum")
)
cross["failure_rate"] = (cross["failures"] / cross["total"]).round(3)
print("=== Failure Rate by Model × Difficulty ===")
display(cross.pivot_table(values="failure_rate",
                          index="model_name",
                          columns="difficulty").round(3))

In [ ]:
# Sample of the 200 failures with summaries — ready for manual annotation
manual_cols = [
    "Puzzle_Number", "Question_Number", "model_name", "difficulty",
    "Solved_Percentile", "audited_answer", "GT_Answer",
    "Reasoning", "error_summary"
]
available = [c for c in manual_cols if c in all_summaries.columns]
manual_df = all_summaries[available].copy()

# Add blank column for your manual coding
manual_df["manual_error_category"] = ""
manual_df["manual_notes"]          = ""

manual_out = "manual_annotation_sheet.csv"
manual_df.to_csv(manual_out, index=False)
print(f"Manual annotation sheet saved to {manual_out}")
display(manual_df.head(5))

## 7. Summary of Outputs

| File | Contents |
|------|----------|
| `failures_sample.csv` | 200 stratified failure rows (raw) |
| `failure_summaries.csv` | Same 200 rows + `error_summary` column from o3 |
| `manual_annotation_sheet.csv` | Summaries + blank columns for your manual coding |
| `error_taxonomy.txt` | Full qualitative taxonomy from o3 |

**Next steps:**  
1. Open `manual_annotation_sheet.csv` and code a random subset (~30–40) yourself  
2. Compare your codes to o3's taxonomy categories — note agreements and gaps  
3. Use the taxonomy + your manual codes to write the error analysis section  